# ACCESS-IoT · Notebook 1 — splits, data views, fidelity (E1), validity (E2), stability (E3)
Input: the released builds (Zenodo, DOI 10.5281/zenodo.23838631), unzipped into `DATA_DIR/builds/<build>/`.
Splits, views and history features are already part of the release. This notebook shows how they were derived:
- **Splits** (V1): re-derived into `REPORTS/rederived/` and compared with the released `splits.parquet`.
- **Views** (V2): re-derived into `REPORTS/rederived/` only if `RECOMPUTE_VIEWS = True`.
- **History features** (V4): causality check of the released `features/history.parquet` (recomputed on a 40-day prefix).
- **E1** workflow statistics for the fidelity table (literature values are added in the paper).
- **E2** validity: single-attribute leakage, train/test drift (adversarial validation), history gain.
- **E3** stability of E1/E2 across the five seed replicates.

Runtime: a GPU is recommended (XGBoost); without one everything runs on the CPU, more slowly. High RAM recommended.

In [ ]:
# V0 — settings
import os, json, time, itertools, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
# Data: unzip every ACCESS-IoT-v1.0_<build>.zip from Zenodo (https://doi.org/10.5281/zenodo.23838631) into DATA_DIR,
# which gives DATA_DIR/builds/<build>/requests.parquet, ... . Outputs (tables, figures) go to REPORTS.
# On Colab: from google.colab import drive; drive.mount("/content/drive") and set DATA_DIR to the folder on your Drive.
DATA_DIR = Path(os.environ.get("ACCESS_IOT_DATA", "data"))
REPORTS = Path(os.environ.get("ACCESS_IOT_REPORTS", "reports"))
REL, REP = DATA_DIR / "builds", REPORTS
REP.mkdir(parents=True, exist_ok=True)
assert REL.exists(), f"no builds found in {REL}: download and unzip the Zenodo files first"
DER = REP / "rederived"                      # re-derived splits / views are written here, never over the released files
RECOMPUTE_VIEWS = False
BUILDS = sorted(p.name for p in REL.iterdir() if (p / "requests.parquet").exists())
SEEDS = [b for b in BUILDS if b.startswith("seed_")]
N_JOBS = min(8, os.cpu_count() or 2)
import xgboost as xgb, torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("builds:", BUILDS, "| xgboost", xgb.__version__, "on", DEVICE, "| parallel jobs:", N_JOBS)
DECISIONS = ["PERMIT", "LIMITED", "DENY", "BREAK_GLASS"]
VISIBLE_KINDS = ["request", "pip", "telemetry", "baseline"]
ID_COLS = {"RequestID", "Timestamp", "SessionID", "UserID", "PatientID", "DeviceID", "DelegationID", "DelegatorID", "DelegateeID"}

def load(build, columns=None):
    d = pd.read_parquet(REL / build / "requests.parquet", columns=columns)
    for c in ("Timestamp", "FeedbackTime"):
        if c in d: d[c] = pd.to_datetime(d[c])
    return d

def dictionary(build):
    return pd.read_csv(REL / build / "data_dictionary.csv")

In [ ]:
# V1 — standard splits (temporal; leave-one-threat-out; department shift chosen once on the main build)
main = load("main", ["RequestID", "Split", "ThreatType", "IsMalicious", "UserDepartment"])
THREATS = sorted(t for t in main.ThreatType.unique() if t != "Benign")
DEPTS = sorted(main.UserDepartment.unique())

def dept_shift_choice(d):
    best = None
    for k in (1, 2):
        for B in itertools.combinations(DEPTS, k):
            te = (d.Split == "test") & d.UserDepartment.isin(B) & (d.IsMalicious == 1)
            tr = (d.Split == "train") & ~d.UserDepartment.isin(B) & (d.IsMalicious == 1)
            cov = min(min((te & (d.ThreatType == t)).sum(), (tr & (d.ThreatType == t)).sum()) for t in THREATS)
            share = (d.Split.eq("test") & d.UserDepartment.isin(B)).sum() / d.Split.eq("test").sum()
            score = (cov, -abs(share - 0.3))                     # every threat covered, about 30% of test traffic
            if best is None or score > best[0]: best = (score, B)
    return list(best[1]), best[0]
TEST_DEPTS, (min_cov, _) = dept_shift_choice(main)
print("department-shift test departments:", TEST_DEPTS, "| min malicious rows per threat (train/test):", min_cov)
json.dump({"temporal_days": {"train": [0, 99], "val": [100, 119], "test": [120, 149]}, "threats": THREATS,
           "dept_shift_test_departments": TEST_DEPTS,
           "actor_disjoint": "per build, the emergency-misuse actors (sorted by UserID) are split alternately into seen/held-out; "
                             "seen actors' fraud appears only in train/val, held-out actors' fraud only in test (T2 official split)"},
          open(REP / "splits_spec.json", "w"), indent=1)

def write_splits(build):
    d = load(build, ["RequestID", "Split", "ThreatType", "UserDepartment", "UserID"])
    s = pd.DataFrame({"RequestID": d.RequestID, "temporal": d.Split})
    for t in THREATS:                                            # LOTO: every row of threat t removed from training
        s[f"loto_{t}"] = np.where((d.Split == "train") & (d.ThreatType == t), "excluded", d.Split)
    inB = d.UserDepartment.isin(TEST_DEPTS)
    s["dept_shift"] = np.select([(d.Split == "train") & ~inB, (d.Split == "val") & ~inB, (d.Split == "test") & inB],
                                ["train", "val", "test"], "excluded")
    # actor-disjoint split for T2: test fraud comes only from emergency-misuse actors never seen committing fraud before
    em = (d.ThreatType == "emergency_misuse").to_numpy()
    actors = sorted(d.UserID[em].unique()); held = d.UserID.isin(actors[1::2]).to_numpy()
    s["actor_disjoint"] = np.where(em & held & (d.Split != "test"), "excluded",
                          np.where(em & ~held & (d.Split == "test"), "excluded", d.Split))
    (DER / build).mkdir(parents=True, exist_ok=True); s.to_parquet(DER / build / "splits.parquet", index=False)
    rel = REL / build / "splits.parquet"
    same = bool(s.reset_index(drop=True).equals(pd.read_parquet(rel)[s.columns].reset_index(drop=True))) if rel.exists() else None
    return build, {**s.dept_shift.value_counts().to_dict(), "fraud actors seen/held-out": f"{len(actors) - len(actors[1::2])}/{len(actors[1::2])}",
                   "identical to released splits": same}

from joblib import Parallel, delayed
for b, vc in Parallel(n_jobs=N_JOBS)(delayed(write_splits)(b) for b in BUILDS): print(f"{b:16s} {vc}")

In [ ]:
# V2 — data views (ABAC, RBAC, CapBAC, trust/sequence, RL environment)
REWARD = {"description": "immediate reward r(a, a*, m, zeta), zeta = 1 + sensitivity/4",
          "rows": [["any", "a*=a", "+1"], ["LIMITED", "PERMIT or BREAK_GLASS", "+0.5"], ["PERMIT or BREAK_GLASS", "LIMITED", "-zeta"],
                   ["PERMIT or BREAK_GLASS", "DENY", "-5 zeta if malicious else -2 zeta"], ["LIMITED", "DENY", "-2.5 zeta if malicious else -zeta"],
                   ["DENY", "PERMIT or LIMITED", "-1"], ["DENY", "BREAK_GLASS", "-4"]],
          "deferred_decisions": "BREAK_GLASS; LIMITED under an emergency claim; approval (PERMIT/LIMITED) of a delegation grant",
          "reconciliation": {"horizon_days": 21, "no_news_reward": 0.5}}

def governance_mask(d, clinical):
    m = np.zeros((len(d), 4), bool)
    permit = (d.ABACDecision == "Permit").to_numpy() & ~((d.RequestType == "DelegationGrant") & (d.Layer == "Object")).to_numpy()
    bg = ((d.EmergencyFlag == 1) & d.UserRole.isin(clinical) & (d.CapabilityCheck == "Pass") & (d.ABACDecision == "Deny")
          & (d.RequestType == "Access") & (d.Layer != "Object")).to_numpy()
    m[:, 0] = permit; m[:, 1] = permit | bg; m[:, 2] = True; m[:, 3] = bg
    return m

def feedback_events(d):
    fb = d[d.FeedbackLabel.notna()].copy()
    fb["Kind"] = np.where(fb.FeedbackLabel == "CompromisedPath", "device_detect",
                 np.where(fb.FeedbackLabel == "DelegationMisused", "delegation_outcome",
                 np.where((fb.EmergencyFlag == 1) & (fb.ABACDecision == "Deny"), "bg_audit",
                 np.where(fb.ThreatType == "virtual_object_tampering", "device_detect", "incident"))))
    fb["AppliesIf"] = np.select([fb.Kind == "device_detect", fb.Kind == "bg_audit", fb.Kind == "delegation_outcome"],
                                ["always", "break_glass_granted", "delegation_approved"], "request_granted")
    return fb[["RequestID", "FeedbackTime", "FeedbackLabel", "Kind", "AppliesIf", "UserID", "DeviceID", "DelegationID",
               "DelegatorID"]].sort_values("FeedbackTime").reset_index(drop=True)

def write_views(build):
    root = REL / build; v = DER / build / "views"; v.mkdir(parents=True, exist_ok=True)
    d, dic, pol = load(build), dictionary(build), json.load(open(root / "policy.json"))
    users, deleg = pd.read_parquet(root / "users.parquet"), pd.read_parquet(root / "delegations.parquet")
    vis = [c for c in dic.loc[dic.Visibility.isin(VISIBLE_KINDS), "Column"] if c in d]
    # ABAC view: decision-time attributes only
    d[vis + ["Split"]].to_parquet(v / "abac_view.parquet", index=False)
    # RBAC view
    users[["UserID", "UserRole", "Department", "Seniority", "Shift", "MaxDelegationDepth"]].to_csv(v / "rbac_user_roles.csv", index=False)
    pd.DataFrame(pol["permissions"], columns=["Role", "Resource", "Action"]).to_csv(v / "rbac_role_permissions.csv", index=False)
    # CapBAC view: base capability tokens (role permissions) + delegated tokens with chains; revocations from revealed misuse
    perms = pd.DataFrame(pol["permissions"], columns=["UserRole", "Resource", "Action"])
    base = users[["UserID", "UserRole"]].merge(perms, on="UserRole")
    base = base.groupby("UserID").apply(lambda g: pd.Series({"Rights": sorted(f"{r}:{a}" for r, a in zip(g.Resource, g.Action))})).reset_index()
    base = base.assign(TokenID="BASE:" + base.UserID, Holder=base.UserID, Issuer="PolicyAuthority", PatientID=None,
                       ParentTokenID=None, Depth=0, ValidFrom=d.Timestamp.min(), ValidTo=d.Timestamp.max(), Kind="base")
    pr = [x for x in pol["resources"] if x in pol["patient_resources"]]
    dl = deleg.assign(TokenID=deleg.DelegationID, Holder=deleg.DelegateeID, Issuer=deleg.GrantorID,
                      ParentTokenID=np.where(deleg.ParentDelegationID.notna(), deleg.ParentDelegationID, "BASE:" + deleg.GrantorID),
                      ValidFrom=deleg.StartTime, ValidTo=deleg.EndTime, Kind="delegated",
                      Rights=[sorted(f"{r}:{a}" for r in pr for a in pol["scope_actions"][s]) for s in deleg.Scope])
    cols = ["TokenID", "Kind", "Holder", "Issuer", "PatientID", "Rights", "ParentTokenID", "Depth", "ValidFrom", "ValidTo"]
    tokens = pd.concat([base[cols], dl[cols]], ignore_index=True)
    tokens["Rights"] = tokens.Rights.apply(lambda r: ";".join(r))
    tokens.to_parquet(v / "capbac_tokens.parquet", index=False)
    rev = d[d.FeedbackLabel == "DelegationMisused"][["DelegationID", "FeedbackTime"]].rename(
        columns={"DelegationID": "TokenID", "FeedbackTime": "RevealTime"})
    rev.assign(Reason="delegation misuse revealed").to_parquet(v / "capbac_revocation_evidence.parquet", index=False)
    pd.DataFrame({"RequestID": d.RequestID, "TokenID": np.where(d.DelegationID.notna() & (d.RequestType == "Access"),
                                                                d.DelegationID, "BASE:" + d.UserID)}).to_parquet(v / "capbac_request_token.parquet", index=False)
    # trust / sequence view: decision-time attributes keyed by entity, plus delayed outcomes
    d[["RequestID", "Timestamp", "UserID", "DeviceID", "PatientID", "SessionID", "DelegationID", "DelegatorID"] +
      [c for c in vis if c not in ID_COLS]].to_parquet(v / "sequence_events.parquet", index=False)
    ev = feedback_events(d); ev.to_parquet(v / "feedback_events.parquet", index=False)
    # RL environment view
    np.save(v / "rl_admissible_mask.npy", governance_mask(d, pol["clinical_roles"]))
    json.dump({"actions": DECISIONS, "mask_rules": {"G1": "PERMIT/LIMITED only if static ABAC permits (never a delegation grant from the Object layer)",
               "G2": "BREAK_GLASS (or LIMITED) on an ABAC deny only for an emergency claim by a clinician with a passed capability check, access requests only",
               "G3": "no break-glass from the Object layer", "DENY": "always admissible"},
               "reward": REWARD, "feedback": "feedback_events.parquet: an event is delivered at FeedbackTime only if AppliesIf holds for the action taken",
               "splits": "../splits.parquet"}, open(v / "rl_env_spec.json", "w"), indent=1, default=str)
    return build, {"abac cols": len(vis), "tokens": len(tokens), "revocation evidence": len(rev), "feedback events": len(ev)}

if RECOMPUTE_VIEWS:
    for b, info in Parallel(n_jobs=N_JOBS)(delayed(write_views)(b) for b in BUILDS): print(f"{b:16s} {info}")
else:
    print("views: using the released views/ folders (set RECOMPUTE_VIEWS = True to re-derive them into", DER, ")")

In [ ]:
# V3 — E1 workflow statistics (fidelity table; computed on benign behaviour unless stated)
CLIN, PRES = {"Doctor", "Nurse"}, {"VitalSigns", "MedicalRecord", "Prescription", "Imaging"}

def e1_stats(build):
    d = load(build); deleg = pd.read_parquet(REL / build / "delegations.parquet"); users = pd.read_parquet(REL / build / "users.parquet")
    ben = d[(d.IsMalicious == 0) & (d.RequestType == "Access")]
    cl = ben[ben.UserRole.isin(CLIN) & ben.ResourceType.isin(PRES)]
    day = d.Timestamp.dt.floor("D")
    s = {}
    for r in ["Doctor", "Nurse", "Technician", "Administrator", "Researcher"]:
        x = ben[ben.UserRole == r]
        s[f"requests per working day: {r}"] = x.groupby([x.UserID, x.Timestamp.dt.floor("D")]).size().mean()
    s["clinical patient-record accesses by care-team members (%)"] = 100 * cl.OnCareTeam.mean()
    s["clinical accesses to patients of another department (%)"] = 100 * (1 - cl.SameDepartment.mean())
    s["benign requests within scheduled shift (%)"] = 100 * ben.WithinShift.mean()
    s["benign requests 22:00-06:00 (%)"] = 100 * ben.Timestamp.dt.hour.isin([22, 23, 0, 1, 2, 3, 4, 5]).mean()
    s["read operations among benign accesses (%)"] = 100 * (ben.RequestedAction == "Read").mean()
    s["benign requests denied by static ABAC (%)"] = 100 * (ben.ABACDecision == "Deny").mean()
    s["device from another department (%)"] = 100 * (ben.DeviceDepartment != ben.UserDepartment).mean()
    s["failed logins per request (mean)"] = ben.LoginFailedAttempts.mean()
    s["emergency (break-glass) claims per 1,000 requests"] = 1000 * d.EmergencyFlag.mean()
    s["static break-glass overrides per 1,000 requests"] = 1000 * ((d.StaticBreakGlassDecision == "Permit") & (d.ABACDecision == "Deny")).mean()
    em = d[d.EmergencyFlag == 1]
    s["emergency claims that are fraudulent (%)"] = 100 * em.IsMalicious.mean()
    gen = d[d.GenuineEmergencyNeed == 1]
    s["genuine emergencies with NEWS2 >= 5 (%)"] = 100 * (gen.AcuityScore >= 5).mean()
    s["genuine emergencies with NEWS2 0-4 (prodrome) (%)"] = 100 * (gen.AcuityScore <= 4).mean()
    nonep = d[d.EpisodeID < 0]
    s["requests outside acute episodes with NEWS2 0 (%)"] = 100 * (nonep.AcuityScore == 0).mean()
    s["requests outside acute episodes with NEWS2 >= 5 (%)"] = 100 * (nonep.AcuityScore >= 5).mean()
    docs = users[users.UserRole == "Doctor"].UserID
    wd = ben[ben.UserID.isin(docs)].groupby([ben.UserID, ben.Timestamp.dt.floor("D")]).size().shape[0]
    g = d[d.RequestType == "DelegationGrant"]
    s["delegation grants per doctor working day"] = len(deleg[deleg.Depth == 1]) / max(wd, 1)
    s["re-delegations among grants (%)"] = 100 * (deleg.Depth > 1).mean()
    s["delegation depth 1 / 2 / 3 (%)"] = "/".join(f"{100 * (deleg.Depth == k).mean():.1f}" for k in (1, 2, 3))
    s["median delegation duration (h)"] = ((deleg.EndTime - deleg.StartTime).dt.total_seconds() / 3600).median()
    s["requests under a delegation (%)"] = 100 * d.DelegationID.notna().mean()
    s["malicious requests (%)"] = 100 * d.IsMalicious.mean()
    s["requests per patient per day (mean)"] = d.groupby([d.PatientID, day]).size().mean()
    s["outcomes revealed (% of requests)"] = 100 * d.FeedbackLabel.notna().mean()
    s["median feedback delay (days)"] = ((d.FeedbackTime - d.Timestamp).dt.total_seconds() / 86400).median()
    return build, s

E1 = dict(Parallel(n_jobs=N_JOBS)(delayed(e1_stats)(b) for b in BUILDS))
e1 = pd.DataFrame(E1)
e1.to_csv(REP / "E1_workflow_statistics.csv")
print(e1[[c for c in ["main", "sparse_feedback", "noisy_benign", "hard_fraud"] if c in e1]].to_string())

In [ ]:
# V4 — features shared by E2 and the baselines (single request; causal history), computed per build in parallel
def single_row(d, dic):
    cols = [c for c in dic.loc[dic.Visibility.isin(VISIBLE_KINDS), "Column"] if c in d and c not in ID_COLS]
    X = d[cols].copy()
    X["HourOfDay"] = d.Timestamp.dt.hour
    X["HasDelegation"] = d.DelegationID.notna().astype(int)
    X["DeviceDeptMatchesUser"] = (d.DeviceDepartment == d.UserDepartment).astype(int)
    for c in X.columns:
        if not pd.api.types.is_numeric_dtype(X[c]):
            X[c] = X[c].astype(object).fillna("NA").astype(str).astype("category")
    return X

def roll(d, key, x, window, how="sum"):
    keys = d[key].astype(object).fillna("NA").astype(str).to_numpy()
    order = np.lexsort((d.Timestamp.to_numpy(), keys))
    s = pd.Series(np.asarray(x, float)[order], index=pd.DatetimeIndex(d.Timestamp.to_numpy()[order]))
    g = s.groupby(keys[order], sort=False).rolling(window, closed="left")
    r = (g.sum() if how == "sum" else g.mean()).to_numpy()
    out = np.empty(len(d)); out[order] = r
    return np.nan_to_num(out)

def prior_fb(d, key, label="Malicious"):
    out = np.zeros(len(d)); t = d.Timestamp.to_numpy()
    ev = d[(d.FeedbackLabel == label) & d[key].notna()]
    for k, g in ev.groupby(key):
        times = np.sort(g.FeedbackTime.to_numpy()); idx = np.where(d[key].to_numpy(object) == k)[0]
        out[idx] = np.searchsorted(times, t[idx], side="left")
    return out

def history(d):
    one = np.ones(len(d)); H = pd.DataFrame(index=d.index)
    off = ((d.OnCareTeam == 0) & d.ResourceType.isin(PRES)).astype(float)
    H["u_cnt_1h"] = roll(d, "UserID", one, "1h"); H["u_cnt_24h"] = roll(d, "UserID", one, "24h")
    H["u_offteam_7d"] = roll(d, "UserID", off, "7D", "mean"); H["u_offshift_7d"] = roll(d, "UserID", 1 - d.WithinShift, "7D", "mean")
    H["u_denied_7d"] = roll(d, "UserID", (d.ABACDecision == "Deny").astype(float), "7D")
    H["u_emerg_30d"] = roll(d, "UserID", d.EmergencyFlag.astype(float), "30D")
    H["u_vip_7d"] = roll(d, "UserID", d.PatientVIP.astype(float), "7D")
    H["u_write_7d"] = roll(d, "UserID", d.RequestedAction.isin(["Write", "Modify", "Delete", "Execute"]).astype(float), "7D", "mean")
    H["u_grants_7d"] = roll(d, "UserID", (d.RequestType == "DelegationGrant").astype(float), "7D")
    H["u_new_device"] = (d.groupby(["UserID", "DeviceID"]).cumcount() == 0).astype(int).to_numpy()
    H["u_new_network"] = (d.groupby(["UserID", "SourceNetwork"]).cumcount() == 0).astype(int).to_numpy()
    H["u_new_patient"] = (d.groupby(["UserID", "PatientID"]).cumcount() == 0).astype(int).to_numpy()
    H["u_newpat_24h"] = roll(d, "UserID", H.u_new_patient.astype(float), "24h")
    H["s_idx"] = d.groupby("SessionID").cumcount().to_numpy()
    H["dev_attest_7d"] = roll(d, "DeviceID", (d.FirmwareAttestation == "Fail").astype(float), "7D")
    H["dev_pl_24h"] = roll(d, "DeviceID", d.PacketLoss, "24h", "mean")
    if "NetAnomaly" in d: H["dev_netanom_24h"] = roll(d, "DeviceID", d.NetAnomaly, "24h", "mean")
    gr = d[d.RequestType == "DelegationGrant"]
    pair = pd.Series(gr.groupby(["UserID", "DelegateeID"]).cumcount().to_numpy(), index=gr.DelegationID.to_numpy())
    H["deleg_pair_prior"] = d.DelegationID.map(pair).fillna(-1).to_numpy()
    g7 = pd.Series(H.u_grants_7d.to_numpy()[gr.index], index=gr.DelegationID.to_numpy())
    H["delegator_grants_7d"] = d.DelegationID.map(g7).fillna(0).to_numpy()
    H["u_prior_malicious_fb"] = prior_fb(d, "UserID")
    H["delegator_prior_malicious_fb"] = prior_fb(d, "DelegatorID")
    return H.astype(np.float32)

def check_features(build):
    """Causality check of the released history features: recomputed from the first 40 days only, they must be identical."""
    d = load(build); t0 = time.time()
    H = pd.read_parquet(REL / build / "features/history.parquet")
    pre = d[d.Timestamp < d.Timestamp.min().normalize() + pd.Timedelta(days=40)]
    Hp = history(pre)
    dmax = float(np.abs(Hp.to_numpy() - H[Hp.columns].iloc[pre.index].to_numpy()).max())
    return build, f"{H.shape[1]} history features; 40-day prefix check max|diff| = {dmax:.1e} ({'PASS' if dmax < 1e-6 else 'FAIL'}) in {time.time()-t0:.0f}s"

for b, msg in Parallel(n_jobs=N_JOBS)(delayed(check_features)(b) for b in BUILDS): print(f"{b:16s} {msg}")

In [ ]:
# V5 — E2 validity on every build (GPU gradient boosting)
from sklearn.metrics import roc_auc_score, average_precision_score

def xgb_fit(X, y, Xe, multi=False, seed=0):
    p = dict(n_estimators=400, learning_rate=0.08, max_depth=8, subsample=0.9, colsample_bytree=0.8, tree_method="hist",
             device=DEVICE, enable_categorical=True, max_cat_to_onehot=1, random_state=seed)
    m = (xgb.XGBClassifier(objective="multi:softprob", **p) if multi else xgb.XGBClassifier(**p)).fit(X, y)
    return m, m.predict_proba(Xe)

E2 = {}
for b in BUILDS:
    t0 = time.time(); d, dic = load(b), dictionary(b)
    X1 = single_row(d, dic); H = pd.read_parquet(REL / b / "features/history.parquet"); H.index = X1.index; X2 = pd.concat([X1, H], axis=1)
    tr, te = (d.Split == "train").to_numpy(), (d.Split == "test").to_numpy(); y = d.IsMalicious.to_numpy()
    r = {}
    # (a) single-attribute leakage: train-fitted target encoding for categories, raw value for numbers; AUC on test
    leak = {}
    for c in X1.columns:
        if str(X1[c].dtype) == "category":
            enc = pd.Series(y[tr]).groupby(X1[c].to_numpy()[tr]).mean()
            sc = pd.Series(X1[c].to_numpy()[te]).map(enc).fillna(y[tr].mean()).to_numpy(float)
        else:
            sc = X1[c].to_numpy(float)[te]
        a = roc_auc_score(y[te], np.nan_to_num(sc)); leak[c] = max(a, 1 - a)
    leak = pd.Series(leak).sort_values(ascending=False); leak.to_csv(REP / f"E2_leakage_{b}.csv")
    r["max single-attribute AUC"], r["most informative attribute"] = round(leak.iloc[0], 3), leak.index[0]
    # (b) adversarial validation: can a model tell training rows from test rows? (benign rows, no time features).
    #     Raw flow features are excluded: training and test windows come from disjoint capture files by design.
    Xa = X1.drop(columns=["HourOfDay"] + [c for c in X1 if c.startswith("NF_")]); ben = (y == 0)
    ia = np.flatnonzero(ben & (tr | te)); rs = np.random.default_rng(0).permutation(ia); half = len(rs) // 2
    m, p = xgb_fit(Xa.iloc[rs[:half]], te[rs[:half]].astype(int), Xa.iloc[rs[half:]])
    r["train-vs-test AUC (benign)"] = round(roc_auc_score(te[rs[half:]], p[:, 1]), 3)
    # (c) history gain and decision difficulty
    for nm, X in [("single-request", X1), ("with history", X2)]:
        m, p = xgb_fit(X[tr], y[tr], X[te])
        r[f"PR-AUC {nm}"] = round(average_precision_score(y[te], p[:, 1]), 3)
        r[f"ROC-AUC {nm}"] = round(roc_auc_score(y[te], p[:, 1]), 3)
    r["history gain (PR-AUC)"] = round(r["PR-AUC with history"] - r["PR-AUC single-request"], 3)
    r["checks: no giveaway (<0.95) / single insufficient (<0.90) / history helps (>0.05)"] = (
        f"{'PASS' if r['max single-attribute AUC'] < 0.95 else 'FAIL'} / {'PASS' if r['PR-AUC single-request'] < 0.90 else 'FAIL'} / "
        f"{'PASS' if r['history gain (PR-AUC)'] > 0.05 else 'FAIL'}")
    E2[b] = r; print(f"{b:16s} done in {time.time()-t0:.0f}s")
e2 = pd.DataFrame(E2); e2.to_csv(REP / "E2_validity.csv"); print(e2.to_string())

In [ ]:
# V6 — E3 stability across the seed replicates (main = seed 42 is included)
rep = ["main"] + SEEDS
num1 = e1[rep].apply(pd.to_numeric, errors="coerce").dropna(how="all")
num2 = e2[rep].apply(pd.to_numeric, errors="coerce").dropna(how="all")
stab = pd.concat([num1, num2])
E3 = pd.DataFrame({"mean": stab.mean(1), "sd": stab.std(1), "cv %": 100 * stab.std(1) / stab.mean(1).abs(), "min": stab.min(1), "max": stab.max(1)})
E3.round(4).to_csv(REP / "E3_stability.csv"); print(E3.round(3).to_string())

In [ ]:
# V7 — Fig. 2 panels a-c (validity) for the main build and variants; panel d (telemetry) is added in Notebook 3
import matplotlib.pyplot as plt, textwrap
NAME = {"main": "Main", "sparse_feedback": "Sparse", "noisy_benign": "Noisy-\nbenign", "hard_fraud": "Hard-\nfraud",
        "no_telemetry": "Sim", "wustl_telemetry": "WUSTL"}
fig, ax = plt.subplots(1, 3, figsize=(14, 4.2), gridspec_kw={"width_ratios": [1, 1, 1.35]})
lk = pd.read_csv(REP / "E2_leakage_main.csv", index_col=0).iloc[:, 0].head(15)[::-1]
ax[0].barh(lk.index, lk.values, color="#1f4e79"); ax[0].axvline(0.95, ls="--", c="grey"); ax[0].set_xlim(0.5, 1)
ax[0].set_xlabel("AUC", fontsize=8); ax[0].set_title("(a) single-attribute AUC (test)", fontsize=9)
bb = [b for b in NAME if b in e2]
x = np.arange(len(bb)); w = 0.38
ax[1].bar(x - w / 2, [float(e2.loc["PR-AUC single-request", b]) for b in bb], w, label="single request", color="#9aa5b1")
ax[1].bar(x + w / 2, [float(e2.loc["PR-AUC with history", b]) for b in bb], w, label="with history", color="#1f4e79")
ax[1].set_xticks(x); ax[1].set_xticklabels([NAME[b] for b in bb]); ax[1].set_ylim(0, 1)
ax[1].set_xlabel("ACCESS-IoT configuration", fontsize=8); ax[1].set_ylabel("PR-AUC", fontsize=8)
ax[1].legend(fontsize=7, frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.25), ncol=2)
ax[1].set_title("(b) malicious-request PR-AUC", fontsize=9)
cv = E3["cv %"].replace([np.inf, -np.inf], np.nan).dropna().sort_values().tail(12)
ax[2].barh(["\n".join(textwrap.wrap(i, 38)) for i in cv.index], cv.values, color="#1f4e79")
ax[2].set_xlabel("coefficient of variation (%)", fontsize=8); ax[2].set_title("(c) variation across 6 seeds", fontsize=9)
for a in ax: a.spines[["top", "right"]].set_visible(False); a.tick_params(labelsize=7)
fig.tight_layout(); fig.savefig(REP / "fig2_validity_abc.pdf", bbox_inches="tight"); fig.savefig(REP / "fig2_validity_abc.png", dpi=200, bbox_inches="tight"); plt.show()
print("NEXT: Notebook 2 (reference baselines by access-control approach)")